# 適合率・再現率・F1 を自分で計算する (micro と macro)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_evaluation/prf_manual.ipynb)

NLP クックブック 13-07 ／ 評価指標と統計的検定

ライブラリに頼らず、適合率・再現率・F1 を定義から計算して、micro 平均と macro 平均の違いを理解したい。

In [1]:
from collections import Counter

gold = ["A", "A", "A", "A", "A", "A", "B", "B", "B", "C"]
pred = ["A", "A", "A", "A", "A", "B", "B", "B", "A", "A"]

labels = sorted(set(gold) | set(pred))
tp, fp, fn = Counter(), Counter(), Counter()
for g, p in zip(gold, pred):
    if g == p:
        tp[g] += 1
    else:
        fp[p] += 1     # p と予測したが違った
        fn[g] += 1     # g を見逃した

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

per_class = {}
for c in labels:
    per_class[c] = prf(tp[c], fp[c], fn[c])
    print(f"{c}: P = {per_class[c][0]:.3f}  R = {per_class[c][1]:.3f}  F1 = {per_class[c][2]:.3f}  (TP {tp[c]}, FP {fp[c]}, FN {fn[c]})")

macro_f1 = sum(f for _, _, f in per_class.values()) / len(labels)        # クラスごとの F1 の単純平均
micro = prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))         # 全クラスの TP/FP/FN を合算してから計算
print(f"macro-F1 = {macro_f1:.3f}   micro-F1 = {micro[2]:.3f}   正解率 = {sum(tp.values()) / len(gold):.3f}")

A: P = 0.714  R = 0.833  F1 = 0.769  (TP 5, FP 2, FN 1)
B: P = 0.667  R = 0.667  F1 = 0.667  (TP 2, FP 1, FN 1)
C: P = 0.000  R = 0.000  F1 = 0.000  (TP 0, FP 0, FN 1)
macro-F1 = 0.479   micro-F1 = 0.700   正解率 = 0.700


In [2]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f, _ = precision_recall_fscore_support(gold, pred, average="macro", zero_division=0)
print("scikit-learn の macro:", round(p, 3), round(r, 3), round(f, 3))

# 抽出タスク (固有表現抽出など、正解と予測が「集合」の場合) の F1
gold_set, pred_set = {("山田", "PER"), ("京都", "LOC"), ("ソニー", "ORG")}, {("山田", "PER"), ("京都", "ORG"), ("東京", "LOC")}
n = len(gold_set & pred_set)
print("抽出の P/R/F1:", [round(x, 3) for x in prf(n, len(pred_set) - n, len(gold_set) - n)])

scikit-learn の macro: 0.46 0.5 0.479
抽出の P/R/F1: [0.333, 0.333, 0.333]


- 単一ラベルの多クラス分類では、micro-F1 は正解率と一致する。
- macro-F1 は少数クラス (ここでは一度も当たっていない C) の影響を強く受ける。クラスの偏りがあるときに、モデルの弱点が見える。
- 「macro-F1」には「クラスごとの F1 の平均」と「macro 適合率と macro 再現率の調和平均」の 2 つの流儀があり、値が違う。前者が一般的。